# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
from pathlib import Path

import platform



import numpy as np

import pandas as pd

import sklearn

from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier

from sklearn.impute import SimpleImputer

from sklearn.inspection import permutation_importance

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import average_precision_score, balanced_accuracy_score

from sklearn.model_selection import GroupShuffleSplit

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler



SEED = 42

TOP_KS = (20, 50, 100)



# Locate the starter release from either the repo root or work/notebooks/.

root = Path.cwd()

for _ in range(6):

    if (root / "data" / "raw" / "content_refresh_anonymized.csv").exists():

        break

    root = root.parent

RAW = root / "data" / "raw" / "content_refresh_anonymized.csv"

assert RAW.exists(), "Starter CSV not found; open this notebook from inside the repo."



df = pd.read_csv(RAW)

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)



CONTEXT = ["content_id", "client_id"]

LABEL_OR_SOURCE = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d"]

EXCLUDED = ["provider_used", "model_used", "clicks_last_30d", "sessions_last_30d",

            "clicks_prev_30d", "sessions_prev_30d"]

FEATURES = [column for column in df.columns

            if column not in CONTEXT + LABEL_OR_SOURCE + EXCLUDED + ["is_declining_label"]]



forbidden = set(LABEL_OR_SOURCE + EXCLUDED + CONTEXT + ["is_declining_label"])

assert not (set(FEATURES) & forbidden), "Leakage or identifier entered the feature set."

assert df["content_id"].is_unique and df["client_id"].nunique() == 32



def precision_at_k(labels, scores, k):

    k = min(k, len(labels))

    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")[:k]

    return float(np.asarray(labels)[order].mean())



def baseline_scores(frame):

    impressions = frame["impressions_90d"].fillna(0)

    position = frame["avg_position"].fillna(0)

    ctr = frame["ctr"].fillna(0)

    word_count = frame["word_count"].fillna(0)

    update_age = frame["days_since_last_update"].fillna(0)

    content_age = frame["content_age_days"].fillna(0)



    problem_score = (

        2 * ((update_age >= 180) & (impressions >= 500)).astype(int)

        + 2 * ((impressions >= 500) & position.between(0, 20, inclusive="right") & (position > 0) & (ctr < 0.5)).astype(int)

        + ((word_count > 0) & (word_count < 1200) & (impressions >= 250)).astype(int)

        + ((position > 0) & (position <= 10) & (content_age >= 180)).astype(int)

    )

    visibility = impressions.rank(pct=True)

    return np.where(problem_score > 0, problem_score + visibility, 0.0)



print(f"Loaded {len(df):,} pages from {df['client_id'].nunique()} pseudonymous clients.")

print(f"Target base rate: {df['is_declining_label'].mean():.3f} | safe features: {len(FEATURES)}")

print(f"Python {platform.python_version()} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.